# Chapter 5 — Milvus with Wikipedia Dataset

## 1. Explore the Dataset

In [2]:
from datasets import load_dataset
import polars as pl

DS = "Cohere/wikipedia-2023-11-embed-multilingual-v3"

wiki_ds = load_dataset(DS, 'en', split="train", streaming=True)
print(wiki_ds.features)

Resolving data files:   0%|          | 0/415 [00:00<?, ?it/s]

{'_id': Value('string'), 'url': Value('string'), 'title': Value('string'), 'text': Value('string'), 'emb': List(Value('float32'))}


## 2. Load 1000 Rows

In [3]:
rows = []
for i, row in enumerate(wiki_ds):
    if i >= 1000:
        break
    rows.append(row)

df = pl.DataFrame(rows)
print(df.shape)
df.head(3)

(1000, 5)


_id,url,title,text,emb
str,str,str,str,list[f64]
"""20231101.en_13194570_0""","""https://en.wikipedia.org/wiki/…","""British Arab Commercial Bank""","""The British Arab Commercial Ba…","[0.001785, -0.040497, … -0.03949]"
"""20231101.en_13194570_1""","""https://en.wikipedia.org/wiki/…","""British Arab Commercial Bank""","""BACB has a head office in Lond…","[0.005188, -0.026306, … -0.015717]"
"""20231101.en_13194570_2""","""https://en.wikipedia.org/wiki/…","""British Arab Commercial Bank""","""The bank provides services of …","[0.008186, -0.006195, … 0.001957]"


## 3. Basic Collection (id + emb only)

In [4]:
from pymilvus import MilvusClient, DataType

client = MilvusClient(uri="./milvus.db")

schema = MilvusClient.create_schema()
schema.add_field(field_name="id", datatype=DataType.VARCHAR, is_primary=True, max_length=128)
schema.add_field(field_name="emb", datatype=DataType.FLOAT_VECTOR, dim=1024)

client.create_collection(collection_name="coll_wiki", schema=schema)

## 4. Full Collection with Dynamic Fields

Connect with proper deployed Milvus.

In [9]:
schema2 = MilvusClient.create_schema(enable_dynamic_field=True)
schema2.add_field(field_name="id", datatype=DataType.VARCHAR, is_primary=True, max_length=128)
schema2.add_field(field_name="emb", datatype=DataType.FLOAT_VECTOR, dim=1024)
schema2.add_field(field_name="title", datatype=DataType.VARCHAR, max_length=256)
schema2.add_field(field_name="url", datatype=DataType.VARCHAR, max_length=65535)
schema2.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=65535)

index = MilvusClient.prepare_index_params(field_name="emb", index_type="AUTOINDEX")

client = MilvusClient(uri="http://localhost:19530")
client.create_collection(collection_name="coll_wiki_2", schema=schema2, index_params=index)

pk = df['_id'][0]
entity = {
    "id": pk,
    "emb": df['emb'][0],
    "title": df['title'][0],
    "text": df['text'][0],
    "url": df['url'][0],
    "extra": "This will go into dynamic field"
}

client.insert(collection_name="coll_wiki_2", data=[entity])
res = client.get("coll_wiki_2", pk, output_fields=["id", "title", "text", "url", "extra"])
print(res)

data: ["{'id': '20231101.en_13194570_0', 'title': 'British Arab Commercial Bank', 'text': 'The British Arab Commercial Bank PLC (BACB) is an international wholesale bank incorporated in the United Kingdom that is authorised by the Prudential Regulation Authority (PRA) and regulated by the PRA and the Financial Conduct Authority (FCA). It was founded in 1972 as UBAF Limited, adopted its current name in 1996, and registered as a public limited company in 2009. The bank has clients trading in and out of developing markets in the Middle East and Africa.', 'url': 'https://en.wikipedia.org/wiki/British%20Arab%20Commercial%20Bank', 'extra': 'This will go into dynamic field'}"], extra_info: {}


## 5. Partition API

In [11]:
client.create_partition(collection_name="coll_wiki_2", partition_name="p1")
client.create_partition(collection_name="coll_wiki_2", partition_name="p2")

entity1 = {
    "id": df['_id'][1],
    "emb": df['emb'][1],
    "title": df['title'][1],
    "text": df['text'][1],
    "url": df['url'][1],
    "extra": "This entity is in partition p1"}
res1 = client.insert(collection_name="coll_wiki_2", data=[entity1], partition_name="p1")
print(f"Inserted into p1: {res1}")

entity2 = {
    "id": df['_id'][2],
    "emb": df['emb'][2],
    "title": df['title'][2],
    "text": df['text'][2],
    "url": df['url'][2],
    "extra": "This entity is in partition p2"}
res2 = client.insert(collection_name="coll_wiki_2", data=[entity2], partition_name="p2")
print(f"Inserted into p2: {res2}")

Inserted into p1: {'insert_count': 1, 'ids': ['20231101.en_13194570_1']}
Inserted into p2: {'insert_count': 1, 'ids': ['20231101.en_13194570_2']}
